# 04 — Generalized One-Turn-Delayed KV-Cache Compaction

An online, fixed-ratio Token Eviction (TE) pipeline that works for **any number of turns**.

**At the end of assistant turn `t` (for `t ≥ 2`):** capture the new assistant's post-RoPE queries, score the **previous assistant turn** using causally masked attention normalized over **all physically available keys**, then evict low-scoring KV entries **only from that previous assistant span**. Earlier compacted turns are frozen. The newest assistant turn remains intact until the following turn.

**Methodological choices:** one conversation, greedy decoding, Qwen2.5-0.5B-Instruct, 50% fixed ratio applied separately per eligible assistant span and per layer/KV head, original user messages untouched, end-of-turn marker forced to survive, T2+ assistant **content-token** queries as delayed proxies (exclude final assistant EOS query; retain EOS in the cache), RMS scores across proxy tokens and grouped query heads, stable earlier-position tie-breaking.

## 1. Environment Setup

In [1]:
%pip install -q "transformers==5.16.1" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 110.8 MB/s eta 0:00:00


## 2. Imports and Experiment Configuration

In [2]:
import copy
import math
import time
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda"
DTYPE = torch.bfloat16
MAX_NEW_TOKENS = 192
RETENTION_RATIO = 0.50

# T2+ assistant content queries are our default delayed proxy.
# EOS is always generated and cached; this only controls its use as a query.
INCLUDE_EOS_PROXY = False
RUN_VALIDATIONS = True

torch.manual_seed(0)
assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."
assert 0 < RETENTION_RATIO <= 1

## 3. Load Model and Tokenizer

In [3]:
from transformers import __version__ as transformers_version

print("Transformers:", transformers_version)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
).to(DEVICE)
model.eval()

assert tokenizer.eos_token == "<|im_end|>"
assert model.config._attn_implementation == "sdpa", (
    "Single-token masking and diagnostic checks below assume SDPA."
)
assert model.config.num_attention_heads % model.config.num_key_value_heads == 0

NUM_LAYERS = model.config.num_hidden_layers
NUM_KV_HEADS = model.config.num_key_value_heads

print("Model:", MODEL_NAME)
print("Device:", model.device, "dtype:", model.dtype)
print("Layers:", NUM_LAYERS)
print("Q heads:", model.config.num_attention_heads)
print("KV heads:", NUM_KV_HEADS)
print("EOS:", repr(tokenizer.eos_token), tokenizer.eos_token_id)

Transformers: 5.16.1


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0 dtype: torch.bfloat16
Layers: 24
Q heads: 14
KV heads: 2
EOS: '<|im_end|>' 151645


## 4. Token Formatting and KV-Cache Utilities

In [4]:
def format_user_turn(user_text, first_turn):
    if first_turn:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_text}],
            tokenize=False,
            add_generation_prompt=True,
        )

    # Later turns must not reinsert the default system message.
    return (
        "\n<|im_start|>user\n"
        + user_text
        + "<|im_end|>\n<|im_start|>assistant\n"
    )


def encode_fragment(text):
    return tokenizer.encode(text, add_special_tokens=False)


def cache_length(cache):
    return 0 if cache is None else cache.get_seq_length()


def cache_bytes(cache):
    if cache is None:
        return 0
    return sum(
        layer.keys.numel() * layer.keys.element_size()
        + layer.values.numel() * layer.values.element_size()
        for layer in cache.layers
    )


def synchronize():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def assert_cache_shapes(cache):
    if cache is None:
        return
    length = cache.get_seq_length()
    for layer in cache.layers:
        assert layer.keys.shape == layer.values.shape
        assert layer.keys.shape[1] == NUM_KV_HEADS
        assert layer.keys.shape[2] == length

## 5. Capture the Actual Q-Projection Output and Apply RoPE

In [5]:
class PostRoPEQueryCapture:
    def __init__(self, model):
        self.model = model
        self.queries = {layer_idx: [] for layer_idx in range(NUM_LAYERS)}
        self.pending_positions = {}
        self.handles = []

    def __enter__(self):
        for layer_idx in range(NUM_LAYERS):
            attn = self.model.model.layers[layer_idx].self_attn

            def attention_pre_hook(module, args, kwargs, idx=layer_idx):
                positions = kwargs.get("position_embeddings")
                if positions is None and len(args) > 1:
                    positions = args[1]
                if positions is None:
                    raise RuntimeError("Missing RoPE position embeddings in attention hook")
                self.pending_positions[idx] = positions

            def projection_hook(module, args, output, idx=layer_idx):
                cos, sin = self.pending_positions.pop(idx)
                batch, seq_len, hidden_dim = output.shape
                num_heads = hidden_dim // self.model.model.layers[idx].self_attn.head_dim
                q = output.view(batch, seq_len, num_heads, -1).transpose(1, 2)
                q_rotated, _ = apply_rotary_pos_emb(q, q, cos, sin)
                self.queries[idx].append(q_rotated.detach().clone())

            self.handles.append(attn.register_forward_pre_hook(
                attention_pre_hook, with_kwargs=True
            ))
            self.handles.append(attn.q_proj.register_forward_hook(projection_hook))
        return self

    def stacked(self):
        assert not self.pending_positions, "Unmatched attention/Q-projection hooks"
        result = {}
        for idx, parts in self.queries.items():
            assert parts, f"No queries captured for layer {idx}"
            result[idx] = torch.cat(parts, dim=2)
        return result

    def __exit__(self, exc_type, exc, tb):
        for handle in self.handles:
            handle.remove()
        self.handles.clear()

## 6. Full-Context Delayed-TE Scoring

In [6]:
def score_previous_assistant(
    cache, queries_by_layer, target_start, target_end,
    proxy_start, proxy_end,
):
    total_length = cache.get_seq_length()
    proxy_count = proxy_end - proxy_start
    candidate_count = target_end - target_start
    assert 0 <= target_start < target_end <= proxy_start < proxy_end <= total_length
    assert candidate_count > 0 and proxy_count > 0

    # These are physical indices in the current cache. RoPE itself uses
    # the separately tracked *logical* positions during model forward.
    query_positions = torch.arange(proxy_start, proxy_end, device=model.device)
    key_positions = torch.arange(total_length, device=model.device)
    causal_mask = key_positions[None, :] <= query_positions[:, None]

    scores_by_layer_head = {}
    target_mass = []  # Descriptive diagnostic only; not an adaptive policy.
    assert set(queries_by_layer) == set(range(len(cache.layers)))

    for layer_idx in range(len(cache.layers)):
        q = queries_by_layer[layer_idx]
        k_all = cache.layers[layer_idx].keys
        num_q_heads = q.shape[1]
        num_kv_heads = k_all.shape[1]
        head_dim = q.shape[-1]
        assert q.shape == (1, num_q_heads, proxy_count, head_dim)
        assert num_q_heads % num_kv_heads == 0
        assert k_all.shape[2] == total_length
        group_size = num_q_heads // num_kv_heads
        layer_scores = []

        for kv_head in range(num_kv_heads):
            q_group = q[:, kv_head * group_size:(kv_head + 1) * group_size]
            k = k_all[0, kv_head]

            # [1, group_size, proxy_count, total_length]
            logits = torch.matmul(q_group.float(), k.float().T) / math.sqrt(head_dim)
            logits = logits.masked_fill(~causal_mask[None, None, :, :], -float("inf"))
            attention = torch.softmax(logits, dim=-1)

            # Slice AFTER softmax; never renormalize within target candidates.
            candidate_attention = attention[..., target_start:target_end]
            scores = candidate_attention.square().mean(dim=(0, 1, 2)).sqrt()
            assert scores.shape == (candidate_count,)
            layer_scores.append(scores)
            target_mass.append(candidate_attention.sum(dim=-1).mean().item())

        scores_by_layer_head[layer_idx] = layer_scores

    return scores_by_layer_head, sum(target_mass) / len(target_mass)

## 7. Select and Compact One Assistant Span

In [7]:
def compact_assistant_span(cache, scores_by_layer_head, start, end,
                           assistant_ids, retention_ratio):
    total_length = cache.get_seq_length()
    original_length = end - start
    assert len(assistant_ids) == original_length
    assert 0 < retention_ratio <= 1

    forced_local = [
        idx for idx, token_id in enumerate(assistant_ids)
        if token_id == tokenizer.eos_token_id
    ]
    budget = max(math.ceil(retention_ratio * original_length), len(forced_local))
    assert budget <= original_length

    prefix = torch.arange(start, device=model.device)
    suffix = torch.arange(end, total_length, device=model.device)
    forced = torch.tensor(forced_local, dtype=torch.long, device=model.device)
    selected_by_layer_head = {}

    for layer_idx in range(NUM_LAYERS):
        layer = cache.layers[layer_idx]
        gathered_keys, gathered_values = [], []
        selected_by_layer_head[layer_idx] = []

        for kv_head, scores in enumerate(scores_by_layer_head[layer_idx]):
            assert scores.shape == (original_length,)
            candidates = scores.clone()
            if forced.numel():
                candidates[forced] = -float("inf")
            num_content = budget - len(forced_local)
            # Stable sort makes equal-score ties reproducible (earlier index wins).
            top = torch.argsort(candidates, descending=True, stable=True)[:num_content]
            selected_local = torch.cat([top, forced]).sort().values
            keep = torch.cat([prefix, start + selected_local, suffix])
            assert selected_local.numel() == budget
            assert torch.unique(selected_local).numel() == budget

            gathered_keys.append(layer.keys[:, kv_head, keep, :])
            gathered_values.append(layer.values[:, kv_head, keep, :])
            selected_by_layer_head[layer_idx].append(keep.detach().cpu())

        layer.keys = torch.stack(gathered_keys, dim=1).contiguous()
        layer.values = torch.stack(gathered_values, dim=1).contiguous()

    removed = original_length - budget
    assert cache.get_seq_length() == total_length - removed
    return {
        "original_length": original_length,
        "retained_length": budget,
        "removed": removed,
        "forced_local": forced_local,
        "keep_indices_by_layer_head": selected_by_layer_head,
    }

## 8. Generalized Delayed-Compaction Pipeline

In [8]:
@dataclass
class TurnRecord:
    number: int
    user_text: str
    assistant_ids: list
    assistant_start: int      # Physical KV index, updated when earlier spans shrink
    assistant_end: int        # Exclusive physical index
    original_assistant_length: int
    assistant_logical_start: int
    user_suffix_tokens: int
    user_suffix_ids: list     # Exact token IDs fed for this new user/header segment
    compacted: bool = False
    retained_length: int = 0
    proxy_mass: float = float("nan")


class DelayedTEPipeline:
    def __init__(self, model, tokenizer, mode="fixed", retention_ratio=0.5,
                 include_eos_proxy=False, compaction_validator=None):
        assert mode in ("full", "fixed")
        self.model = model
        self.tokenizer = tokenizer
        self.mode = mode
        self.retention_ratio = retention_ratio
        self.include_eos_proxy = include_eos_proxy
        self.compaction_validator = compaction_validator
        self.validated_compactions = 0
        self.cache = None
        self.logical_position = 0
        self.turns = []
        self.events = []
        self.logical_token_ids = []  # Complete processed token history for replay validation.

    def _forward_one(self, token_id):
        # q_len=1: the SDPA path attends to every physical cache entry.
        token = torch.tensor([[token_id]], dtype=torch.long, device=model.device)
        next_length = cache_length(self.cache) + 1
        with torch.no_grad():
            output = self.model(
                input_ids=token,
                attention_mask=torch.ones((1, next_length), dtype=torch.long,
                                          device=model.device),
                position_ids=torch.tensor([[self.logical_position]],
                                          dtype=torch.long, device=model.device),
                past_key_values=self.cache,
                use_cache=True,
            )
        self.cache = output.past_key_values
        self.logical_position += 1
        return output

    def _prefill_token_ids(self, token_ids):
        # Reusable entry point for future tool/environment segments.
        assert token_ids, "Empty formatted segment"
        if self.cache is None:
            # Only the first prompt is batched. Later suffixes are processed
            # one token at a time to preserve the validated q_len=1 SDPA path.
            inputs = torch.tensor([token_ids], dtype=torch.long, device=model.device)
            with torch.no_grad():
                output = self.model(
                    input_ids=inputs,
                    attention_mask=torch.ones_like(inputs),
                    position_ids=torch.arange(len(token_ids), device=model.device)[None, :],
                    use_cache=True,
                )
            self.cache = output.past_key_values
            self.logical_position = len(token_ids)
        else:
            for token_id in token_ids:
                output = self._forward_one(token_id)
        self.logical_token_ids.extend(token_ids)
        return output

    def _prefill_user(self, user_text):
        fragment = format_user_turn(user_text, first_turn=not self.turns)
        token_ids = encode_fragment(fragment)
        return self._prefill_token_ids(token_ids), token_ids

    def _generate_assistant(self, previous_output, collect_proxy):
        assistant_start = cache_length(self.cache)
        token_ids = []
        capture = PostRoPEQueryCapture(self.model) if collect_proxy else None

        # No hook for the first turn or full-cache baseline.
        if capture is not None:
            capture.__enter__()
        try:
            next_token = torch.argmax(previous_output.logits[:, -1, :], dim=-1).item()
            for _ in range(MAX_NEW_TOKENS):
                token_ids.append(next_token)
                output = self._forward_one(next_token)
                self.logical_token_ids.append(next_token)
                if next_token == self.tokenizer.eos_token_id:
                    break
                next_token = torch.argmax(output.logits[:, -1, :], dim=-1).item()
            else:
                raise RuntimeError("Assistant did not reach EOS; increase MAX_NEW_TOKENS")
            captured = capture.stacked() if capture is not None else None
        finally:
            if capture is not None:
                capture.__exit__(None, None, None)

        assistant_end = cache_length(self.cache)
        assert assistant_end - assistant_start == len(token_ids)
        return token_ids, assistant_start, assistant_end, captured

    def run_turn(self, user_text):
        previous = self.turns[-1] if self.turns else None
        user_output, suffix_ids = self._prefill_user(user_text)
        suffix_len = len(suffix_ids)
        assistant_logical_start = self.logical_position
        collect_proxy = self.mode == "fixed" and previous is not None
        assistant_ids, assistant_start, assistant_end, captured = (
            self._generate_assistant(user_output, collect_proxy)
        )
        current = TurnRecord(
            number=len(self.turns) + 1,
            user_text=user_text,
            assistant_ids=assistant_ids,
            assistant_start=assistant_start,
            assistant_end=assistant_end,
            original_assistant_length=len(assistant_ids),
            assistant_logical_start=assistant_logical_start,
            user_suffix_tokens=suffix_len,
            user_suffix_ids=suffix_ids,
            retained_length=len(assistant_ids),
        )
        self.turns.append(current)

        if collect_proxy:
            # Only content-token queries are used by default; EOS remains
            # generated/cached. Empty content falls back explicitly to EOS.
            proxy_count = len(assistant_ids)
            used_eos_fallback = False
            if not self.include_eos_proxy:
                if proxy_count > 1:
                    proxy_count -= 1
                else:
                    used_eos_fallback = True
            proxy_end = assistant_start + proxy_count
            proxies = {
                idx: queries[:, :, :proxy_count, :]
                for idx, queries in captured.items()
            }
            synchronize()
            score_start = time.perf_counter()
            scores, target_mass = score_previous_assistant(
                self.cache, proxies,
                previous.assistant_start, previous.assistant_end,
                assistant_start, proxy_end,
            )
            synchronize()
            score_ms = (time.perf_counter() - score_start) * 1000

            # Validation is injected by the notebook's explicitly labeled
            # validation cell. A temporary pre-eviction snapshot is held only
            # for the current event, not accumulated over the conversation.
            before_compaction = (
                copy.deepcopy(self.cache)
                if self.compaction_validator is not None else None
            )

            old_target_end = previous.assistant_end
            synchronize()
            compact_start = time.perf_counter()
            result = compact_assistant_span(
                self.cache, scores, previous.assistant_start,
                previous.assistant_end, previous.assistant_ids,
                self.retention_ratio,
            )
            synchronize()
            compact_ms = (time.perf_counter() - compact_start) * 1000

            if self.compaction_validator is not None:
                self.compaction_validator(
                    before_compaction, self.cache, result,
                    target_turn=previous.number,
                    proxy_turn=current.number,
                )
                self.validated_compactions += 1
            del before_compaction

            previous.compacted = True
            previous.retained_length = result["retained_length"]
            previous.proxy_mass = target_mass

            # All later assistant spans shift left by the number removed.
            previous.assistant_end -= result["removed"]
            for turn in self.turns:
                if turn.number > previous.number:
                    assert turn.assistant_start >= old_target_end
                    turn.assistant_start -= result["removed"]
                    turn.assistant_end -= result["removed"]

            self.events.append({
                "target_turn": previous.number,
                "proxy_turn": current.number,
                "original_tokens": result["original_length"],
                "retained_tokens": result["retained_length"],
                "removed_tokens": result["removed"],
                "mean_target_attention_mass": target_mass,
                "eos_fallback": used_eos_fallback,
                "score_ms_exploratory": score_ms,
                "compaction_ms_exploratory": compact_ms,
                "keep_indices_by_layer_head": result["keep_indices_by_layer_head"],
            })

        assert_cache_shapes(self.cache)
        # Logical positions advance with every processed token, never eviction.
        assert self.logical_position >= cache_length(self.cache)
        assert self.logical_position == len(self.logical_token_ids)
        return {
            "turn": current.number,
            "response": self.tokenizer.decode(assistant_ids, skip_special_tokens=True),
            "assistant_ids": assistant_ids,
            "physical_tokens": cache_length(self.cache),
            "logical_tokens": self.logical_position,
            "kv_mebibytes": cache_bytes(self.cache) / 2**20,
        }

## [VALIDATION A] Independent Scoring Reference and Exact Chat-Token Boundaries
**Assumptions tested:** (1) only causally visible keys enter the full-context softmax; (2) GQA mapping and RMS scores agree with a slower per-query/head reference; (3) incremental encoding of **known multi-turn chat content** matches Qwen's canonical chat-template **token IDs**, not merely the rendered text.

In [9]:
from types import SimpleNamespace

def validate_scoring_and_formatting():
    # ============================================================
    # PART 1: Independent RMS / GQA scoring validation
    # ============================================================

    gen = torch.Generator(device="cpu").manual_seed(13)

    k = torch.randn(1, 2, 6, 4, generator=gen, device="cpu")
    q = torch.randn(1, 4, 2, 4, generator=gen, device="cpu")

    fake_cache = SimpleNamespace(
        layers=[SimpleNamespace(keys=k.to(model.device))],
        get_seq_length=lambda: 6,
    )

    queries = {0: q.to(model.device)}

    actual, _ = score_previous_assistant(
        fake_cache,
        queries,
        target_start=1,
        target_end=3,
        proxy_start=4,
        proxy_end=6,
    )

    for kv_head in range(2):
        samples = []

        for qhead in range(kv_head * 2, (kv_head + 1) * 2):
            for proxy_index, position in enumerate([4, 5]):
                # Only causally available keys participate.
                logits = (
                    q[0, qhead, proxy_index]
                    @ k[0, kv_head, :position + 1].T
                    / 2.0
                )

                probs = torch.softmax(logits, dim=-1)
                samples.append(probs[1:3])

        reference = (
            torch.stack(samples)
            .square()
            .mean(dim=0)
            .sqrt()
        )

        torch.testing.assert_close(
            actual[0][kv_head].cpu(),
            reference,
            atol=1e-6,
            rtol=1e-6,
        )

    print("PASS: independent causal full-context RMS/GQA scores")

    # ============================================================
    # PART 2: Incremental Qwen chat-formatting validation
    # ============================================================

    known_turns = [
        ("Plan a picnic.", "Certainly."),
        ("What is the plan?", "A picnic in the park."),
    ]

    last_user = "Which location did we choose?"

    messages = []
    incremental_ids = []
    incremental_text = ""

    for index, (user_text, assistant_text) in enumerate(known_turns):
        fragment = format_user_turn(
            user_text,
            first_turn=(index == 0),
        )

        # Build token IDs exactly as the pipeline does.
        incremental_ids.extend(encode_fragment(fragment))
        incremental_ids.extend(encode_fragment(assistant_text))
        incremental_ids.append(tokenizer.eos_token_id)

        # Build the equivalent text for debugging.
        incremental_text += fragment
        incremental_text += assistant_text
        incremental_text += tokenizer.eos_token

        messages.extend([
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text},
        ])

    final_fragment = format_user_turn(
        last_user,
        first_turn=False,
    )

    incremental_ids.extend(encode_fragment(final_fragment))
    incremental_text += final_fragment

    messages.append({
        "role": "user",
        "content": last_user,
    })

    # ============================================================
    # PART 3: Obtain canonical Qwen text and token IDs
    # ============================================================

    canonical_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Encode the canonical rendered text directly.
    # This avoids differences in return types across Transformers versions.
    canonical_encoding = tokenizer(
        canonical_text,
        add_special_tokens=False,
    )

    canonical_ids = canonical_encoding["input_ids"]

    if isinstance(canonical_ids, torch.Tensor):
        canonical_ids = canonical_ids.flatten().tolist()
    elif canonical_ids and isinstance(canonical_ids[0], list):
        canonical_ids = canonical_ids[0]

    canonical_ids = [int(token_id) for token_id in canonical_ids]

    # ============================================================
    # PART 4: Compare and diagnose any mismatch
    # ============================================================

    text_matches = incremental_text == canonical_text
    ids_match = incremental_ids == canonical_ids

    print("\nChat formatting validation:")
    print("Text matches:", text_matches)
    print("Token IDs match:", ids_match)
    print("Incremental token count:", len(incremental_ids))
    print("Canonical token count:", len(canonical_ids))

    if not text_matches:
        mismatch = next(
            (
                i
                for i in range(
                    min(len(incremental_text), len(canonical_text))
                )
                if incremental_text[i] != canonical_text[i]
            ),
            min(len(incremental_text), len(canonical_text)),
        )

        left = max(0, mismatch - 35)
        right = mismatch + 80

        print("\nFirst text mismatch at character:", mismatch)
        print(
            "Incremental:",
            repr(incremental_text[left:right]),
        )
        print(
            "Canonical:  ",
            repr(canonical_text[left:right]),
        )

    if not ids_match:
        mismatch = next(
            (
                i
                for i in range(
                    min(len(incremental_ids), len(canonical_ids))
                )
                if incremental_ids[i] != canonical_ids[i]
            ),
            min(len(incremental_ids), len(canonical_ids)),
        )

        left = max(0, mismatch - 5)
        right = mismatch + 8

        incremental_slice = incremental_ids[left:right]
        canonical_slice = canonical_ids[left:right]

        print("\nFirst token mismatch at index:", mismatch)
        print("Incremental IDs:", incremental_slice)
        print("Canonical IDs:  ", canonical_slice)

        print(
            "Incremental tokens:",
            tokenizer.convert_ids_to_tokens(incremental_slice),
        )
        print(
            "Canonical tokens:  ",
            tokenizer.convert_ids_to_tokens(canonical_slice),
        )

    # ============================================================
    # PART 5: Validation result
    # ============================================================

    if text_matches and ids_match:
        print("\nPASS: exact Qwen canonical multi-turn chat token IDs")
    else:
        print("\nFAIL: incremental formatting/tokenization mismatch")
        print(
            "Resolve this mismatch before running later validations."
        )


if RUN_VALIDATIONS:
    validate_scoring_and_formatting()

PASS: independent causal full-context RMS/GQA scores

Chat formatting validation:
Text matches: True
Token IDs match: True
Incremental token count: 72
Canonical token count: 72

PASS: exact Qwen canonical multi-turn chat token IDs


In [10]:
def validate_recorded_chat_boundaries(pipeline):
    # The pipeline stores what was *actually* processed, not a reconstruction
    # from decoded text. All logical assistant offsets must match those IDs.
    cursor = 0
    rendered = ""
    messages = []
    for index, turn in enumerate(pipeline.turns):
        assert turn.number == index + 1
        assert turn.user_suffix_tokens == len(turn.user_suffix_ids)
        assert turn.user_suffix_ids == encode_fragment(
            format_user_turn(turn.user_text, first_turn=(index == 0))
        )
        user_end = cursor + turn.user_suffix_tokens
        assert pipeline.logical_token_ids[cursor:user_end] == turn.user_suffix_ids
        assert turn.assistant_logical_start == user_end
        assistant_end = user_end + len(turn.assistant_ids)
        assert pipeline.logical_token_ids[user_end:assistant_end] == turn.assistant_ids
        assert turn.assistant_ids[-1] == tokenizer.eos_token_id
        assert turn.original_assistant_length == len(turn.assistant_ids)

        # Compare *rendered* full messages to the canonical chat template.
        # Generated token IDs need not round-trip to the same BPE segmentation.
        assistant_text = tokenizer.decode(
            turn.assistant_ids[:-1], skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )
        rendered += format_user_turn(turn.user_text, index == 0)
        rendered += tokenizer.decode(
            turn.assistant_ids, skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )
        messages.extend([
            {"role": "user", "content": turn.user_text},
            {"role": "assistant", "content": assistant_text},
        ])
        cursor = assistant_end

    assert cursor == pipeline.logical_position == len(pipeline.logical_token_ids)
    canonical_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    assert rendered + "\n" == canonical_text, (
        "Rendered runtime transcript does not match Qwen chat formatting"
    )

    # Diagnostic only: a model can emit alternative valid BPE segmentations.
    canonical_without_final_newline = encode_fragment(canonical_text[:-1])
    reencode_matches = (canonical_without_final_newline == pipeline.logical_token_ids)
    if not reencode_matches:
        first_difference = next(
            (i for i, (a, b) in enumerate(zip(
                canonical_without_final_newline, pipeline.logical_token_ids
            )) if a != b),
            min(len(canonical_without_final_newline), len(pipeline.logical_token_ids)),
        )
        print(
            f"DIAGNOSTIC ({pipeline.mode}): re-tokenized generated text differs "
            f"at ID offset {first_difference}; exact recorded boundaries PASS. "
            "Generated BPE paths need not be canonical."
        )
    print(f"PASS ({pipeline.mode}): every user/assistant logical boundary "
          "and Qwen chat-template structure")
    return reencode_matches


## [VALIDATION B] Every-Event Exact K/V Gather and Post-Eviction Continuation
**Assumptions tested:** at **every delayed compaction** (T2 → T1 through T5 → T4), each retained key and value equals the corresponding entry in an independent pre-eviction snapshot, in its original order, for **every layer and KV head**. The saved snapshot is immediately released after that event.

After each compaction, a **one-token probe on a cloned cache** checks that continuation appends one KV entry without changing *any* historical retained K or V, and does not mutate the actual experiment.

In [11]:
def validate_compaction_gather(before, after, result, *, target_turn, proxy_turn):
    # Called by the pipeline at each event, BEFORE another turn can append K/V.
    assert_cache_shapes(before)
    assert_cache_shapes(after)
    assert cache_length(before) - cache_length(after) == result["removed"]
    assert result["retained_length"] + result["removed"] == result["original_length"]
    for layer_idx in range(NUM_LAYERS):
        for kv_head in range(NUM_KV_HEADS):
            keep = result["keep_indices_by_layer_head"][layer_idx][kv_head]
            keep = keep.to(model.device)
            assert keep.ndim == 1 and keep.numel() == cache_length(after)
            if keep.numel() > 1:
                assert torch.all(keep[1:] > keep[:-1])
            assert keep.min().item() >= 0
            assert keep.max().item() < cache_length(before)
            for name in ("keys", "values"):
                original = getattr(before.layers[layer_idx], name)
                compacted = getattr(after.layers[layer_idx], name)
                expected = original[:, kv_head, keep, :]
                observed = compacted[:, kv_head, :, :]
                assert torch.equal(observed, expected), (
                    f"Wrong {name} gather: T{proxy_turn} -> T{target_turn}, "
                    f"layer {layer_idx}, KV head {kv_head}"
                )
    print(f"PASS: exact K/V gather, T{proxy_turn} -> T{target_turn}, "
          f"{result['removed']} entries removed per KV head/layer")


def validate_continuation_preserves_history(pipeline):
    # The probe must operate on a deep copy, never on the live cache.
    original = pipeline.cache
    old_length = cache_length(original)
    cloned = copy.deepcopy(original)
    probe_id = tokenizer.encode(" hello", add_special_tokens=False)[0]
    with torch.no_grad():
        result = model(
            input_ids=torch.tensor([[probe_id]], dtype=torch.long, device=model.device),
            attention_mask=torch.ones((1, old_length + 1), dtype=torch.long,
                                      device=model.device),
            position_ids=torch.tensor([[pipeline.logical_position]],
                                      dtype=torch.long, device=model.device),
            past_key_values=cloned,
            use_cache=True,
        )
    continued = result.past_key_values
    assert cache_length(continued) == old_length + 1
    assert cache_length(pipeline.cache) == old_length
    for layer_idx in range(NUM_LAYERS):
        for name in ("keys", "values"):
            before = getattr(original.layers[layer_idx], name)
            after = getattr(continued.layers[layer_idx], name)
            # This compares *all* KV heads and all historical positions.
            assert torch.equal(after[:, :, :old_length, :], before), (
                f"Historical {name} changed on continuation, layer {layer_idx}"
            )
    del continued, cloned
    print(f"PASS: one-token continuation preserved all historical K/V "
          f"after T{pipeline.turns[-1].number}")

## 9. Controlled Multi-Turn Workload

In [12]:
USER_TURNS = [
    (
        "I'm planning a tea party. The theme is lavender and the special "
        "dessert is lemon cake. Remember this. Draft a short invitation "
        "and leave the time and location as placeholders."
    ),
    "What theme and special dessert did I choose?",
    "Which details did you leave as placeholders in the invitation?",
    "Write a one-sentence reminder of the theme and dessert, without inventing a location.",
    "What was the original dessert, and which event was it for?",
]

full = DelayedTEPipeline(model, tokenizer, mode="full")
fixed = DelayedTEPipeline(
    model, tokenizer, mode="fixed", retention_ratio=RETENTION_RATIO,
    include_eos_proxy=INCLUDE_EOS_PROXY,
    compaction_validator=validate_compaction_gather if RUN_VALIDATIONS else None,
)
full_results, fixed_results = [], []

## 10. Run the Full-Cache and Delayed-TE Pipelines
The loop is the **only** driver for the main experiment. The validation callback checks the exact K/V gather at **every** compaction, immediately when it occurs. Afterward, a clone-only continuation probe checks the preserved historical KV entries. The model's actual running cache is not changed by either validation.

In [13]:
for turn_index, question in enumerate(USER_TURNS, start=1):
    full_result = full.run_turn(question)
    fixed_result = fixed.run_turn(question)
    full_results.append(full_result)
    fixed_results.append(fixed_result)

    # No eviction occurs before the end of T2, so T1/T2 outputs must match.
    if RUN_VALIDATIONS and turn_index <= 2:
        assert full_result["assistant_ids"] == fixed_result["assistant_ids"]

    if RUN_VALIDATIONS and turn_index >= 2:
        validate_continuation_preserves_history(fixed)

    print(f"\n=== TURN {turn_index} ===")
    print("User:", question)
    print("FULL:", full_result["response"])
    print("FIXED:", fixed_result["response"])
    print("Physical KV positions: full / fixed =",
          full_result["physical_tokens"], "/", fixed_result["physical_tokens"])

print("\nCompaction events:", len(fixed.events))
assert len(fixed.events) == len(USER_TURNS) - 1
if RUN_VALIDATIONS:
    assert fixed.validated_compactions == len(fixed.events), (
        "Some compaction events were not independently checked"
    )
    validate_recorded_chat_boundaries(full)
    validate_recorded_chat_boundaries(fixed)


=== TURN 1 ===
User: I'm planning a tea party. The theme is lavender and the special dessert is lemon cake. Remember this. Draft a short invitation and leave the time and location as placeholders.
FULL: Dear [Guest's Name],

I hope this message finds you well. I am writing to invite you to our tea party, which is themed around lavender and will feature a special dessert, the lemon cake. The event is scheduled for [Date and Time] at [Location].

The lavender theme is perfect for a cozy and relaxing atmosphere, and the lemon cake is a delightful choice that will complement the lavender decor. We are looking forward to welcoming you to our tea party and looking forward to a memorable experience together.

Please let us know if you are available to attend, and we will be happy to confirm your attendance. If you have any questions or need further information, please do not hesitate to contact us.

Thank you for considering our invitation. We look forward to seeing you at the tea party.

Be

## 11. Inspect Memory and Per-Turn Retention
This is the initial **full-cache vs fixed-ratio baseline**, not an adaptive policy. The `mean_target_attention_mass` values are **observations**, not a proposed mapping to retention ratios. Times are synchronized exploratory measurements that include Python/hook overhead, **not** comparative serving benchmarks.

In [14]:
import pandas as pd

summary = pd.DataFrame([
    {
        "turn": n,
        "full_physical_tokens": a["physical_tokens"],
        "fixed_physical_tokens": b["physical_tokens"],
        "full_KV_MiB": round(a["kv_mebibytes"], 3),
        "fixed_KV_MiB": round(b["kv_mebibytes"], 3),
        "exact_generated_IDs": a["assistant_ids"] == b["assistant_ids"],
    }
    for n, (a, b) in enumerate(zip(full_results, fixed_results), start=1)
])
display(summary)

events = pd.DataFrame([
    {k: v for k, v in event.items() if k != "keep_indices_by_layer_head"}
    for event in fixed.events
])
display(events)
print("Final cache saved vs full:",
      f"{100 * (1 - cache_bytes(fixed.cache) / cache_bytes(full.cache)):.1f}%")

,turn,full_physical_tokens,fixed_physical_tokens,full_KV_MiB,fixed_KV_MiB,exact_generated_IDs
0,1,231,231,2.707,2.707,True
1,2,269,186,3.152,2.180,True
2,3,311,226,3.645,2.648,False
3,4,366,263,4.289,3.082,False
4,5,406,291,4.758,3.410,True


,target_turn,proxy_turn,original_tokens,retained_tokens,removed_tokens,mean_target_attention_mass,eos_fallback,score_ms_exploratory,compaction_ms_exploratory
0,1,2,167,84,83,0.156620,False,24.463343,125.153723
1,2,3,20,10,10,0.031578,False,16.749120,22.785737
2,3,4,30,15,15,0.029590,False,16.681417,22.754219
3,4,5,25,13,12,0.058379,False,16.796522,23.257642


Final cache saved vs full: 28.3%


## [VALIDATION C / DIAGNOSTIC] Compare Captured Q and GQA Attention with Actual SDPA
**Independent runtime check:** on a cloned cache, process one probe token. Compare our hook's post-RoPE Q to the query **actually passed to PyTorch SDPA**. Reconstruct one attention operation using SDPA's Q/K/V inputs and GQA head replication, then compare its result to SDPA output.

In [15]:
def validate_q_capture_and_sdpa(pipeline):
    assert model.config._attn_implementation == "sdpa"
    original_sdpa = F.scaled_dot_product_attention
    calls = []
    actual_queries = []

    def inspect_sdpa(query, key, value, *args, **kwargs):
        output = original_sdpa(query, key, value, *args, **kwargs)
        # Capture actual Q for every layer; retain full Q/K/V only for one
        # representative layer to avoid unnecessary GPU memory usage.
        actual_queries.append(query.detach().clone())
        if len(calls) == 0:
            calls.append({
                "q": query.detach().clone(),
                "k": key.detach().clone(),
                "v": value.detach().clone(),
                "output": output.detach().clone(),
                "mask": kwargs.get("attn_mask", args[0] if args else None),
                "causal": kwargs.get("is_causal", False),
                "scale": kwargs.get("scale", None),
            })
        return output

    cache_copy = copy.deepcopy(pipeline.cache)
    probe_token_id = tokenizer.encode(" hello", add_special_tokens=False)[0]
    with PostRoPEQueryCapture(model) as capture:
        try:
            F.scaled_dot_product_attention = inspect_sdpa
            with torch.no_grad():
                model(
                    input_ids=torch.tensor([[probe_token_id]], device=model.device),
                    attention_mask=torch.ones(
                        (1, cache_length(cache_copy) + 1),
                        dtype=torch.long, device=model.device,
                    ),
                    position_ids=torch.tensor([[pipeline.logical_position]],
                                              dtype=torch.long, device=model.device),
                    past_key_values=cache_copy,
                    use_cache=True,
                )
        finally:
            F.scaled_dot_product_attention = original_sdpa
        captured = capture.stacked()

    assert len(actual_queries) == NUM_LAYERS, (
        "Expected exactly one SDPA query for every transformer layer"
    )
    assert len(calls) == 1, "No SDPA call intercepted"
    for layer_idx, actual_q in enumerate(actual_queries):
        torch.testing.assert_close(
            captured[layer_idx], actual_q, rtol=0, atol=0
        )
    call = calls[0]
    assert call["mask"] is None and not call["causal"], (
        "Unexpected mask in q_len=1 SDPA; revisit continuation assumptions"
    )

    q = call["q"].float()
    k = call["k"].float()
    v = call["v"].float()
    assert q.shape[1] % k.shape[1] == 0
    group_size = q.shape[1] // k.shape[1]
    k = k.repeat_interleave(group_size, dim=1)
    v = v.repeat_interleave(group_size, dim=1)
    scale = call["scale"] if call["scale"] is not None else 1 / math.sqrt(q.shape[-1])
    weights = torch.softmax((q @ k.transpose(-2, -1)) * scale, dim=-1)
    reference_output = weights @ v
    torch.testing.assert_close(
        reference_output, call["output"].float(),
        atol=0.03, rtol=0.05,
    )
    print("PASS: actual SDPA queries match post-RoPE hook across all layers")
    print("PASS: causal visibility and GQA-reconstructed attention output")


if RUN_VALIDATIONS:
    validate_q_capture_and_sdpa(fixed)

PASS: actual SDPA queries match post-RoPE hook across all layers
PASS: causal visibility and GQA-reconstructed attention output


## [VALIDATION D] Final Cache and Turn-Bookkeeping Invariants
**Assumptions tested:** the logical position counts *all* processed tokens; physical positions shrink only on eviction; every retained assistant span has consistent length; the newest assistant remains intact; the reported event counts reconcile with the full-cache length.

In [16]:
if RUN_VALIDATIONS:
    assert_cache_shapes(full.cache)
    assert_cache_shapes(fixed.cache)
    assert full.logical_position == cache_length(full.cache)
    assert fixed.logical_position == len(fixed.logical_token_ids)
    assert full.logical_position == len(full.logical_token_ids)
    for pipe in (full, fixed):
        assert pipe.logical_position == sum(
            turn.user_suffix_tokens + turn.original_assistant_length
            for turn in pipe.turns
        )
    assert fixed.logical_position - cache_length(fixed.cache) == sum(
        event["removed_tokens"] for event in fixed.events
    )
    assert not fixed.turns[-1].compacted
    assert all(turn.compacted for turn in fixed.turns[:-1])
    for turn in fixed.turns:
        assert turn.assistant_end - turn.assistant_start == turn.retained_length
        assert turn.user_suffix_tokens == len(turn.user_suffix_ids)
        assert turn.assistant_ids[-1] == tokenizer.eos_token_id
    assert all(fixed.turns[i].assistant_end <= fixed.turns[i+1].assistant_start
               for i in range(len(fixed.turns)-1))
    print("PASS: cache shapes, physical/logical positions, frozen-turn spans")

PASS: cache shapes, physical/logical positions, frozen-turn spans


## [VALIDATION E / DIAGNOSTIC] Ordinary Full-Context Replay
Re-feed the full baseline's **exact generated token IDs** as one ordinary, uncompressed model forward. For each assistant token, ask whether the normal full-context model's preceding-position logits choose that same greedy token. This is a check against a conventional batched forward, not merely an internal cache-length assertion.

In [17]:
def validate_full_replay(full_pipeline):
    all_ids = torch.tensor(
        [full_pipeline.logical_token_ids],
        dtype=torch.long,
        device=model.device,
    )

    with torch.no_grad():
        replay = model(
            input_ids=all_ids,
            attention_mask=torch.ones_like(all_ids),
            use_cache=False,
        )

    mismatches = []
    total_checked = 0

    for turn in full_pipeline.turns:
        start = turn.assistant_logical_start
        end = start + len(turn.assistant_ids)

        # Token at position p is predicted by logits at p-1.
        logits = replay.logits[0, start - 1:end - 1].float()
        predicted = logits.argmax(dim=-1).tolist()

        for offset, (got, expected) in enumerate(
            zip(predicted, turn.assistant_ids)
        ):
            total_checked += 1

            if got != expected:
                token_logits = logits[offset]

                expected_logit = token_logits[expected].item()
                winner_logit = token_logits[got].item()
                margin = winner_logit - expected_logit

                mismatches.append({
                    "turn": turn.number,
                    "offset": offset,
                    "replay_token": got,
                    "cached_token": expected,
                    "logit_margin": round(margin, 6),
                })

    del replay

    print("Total generated tokens checked:", total_checked)
    print("Mismatches:", len(mismatches))

    if mismatches:
        print("\nFirst mismatches:")
        for item in mismatches[:10]:
            print(item)

        print(
            "\nDIAGNOSTIC: Cached and full-sequence greedy decoding "
            "disagree. Inspect logit margins before deciding whether "
            "this is numerical sensitivity or a correctness issue."
        )
    else:
        print(
            "PASS: full-context replay agrees with all cached greedy tokens"
        )

    return mismatches


if RUN_VALIDATIONS:
    replay_mismatches = validate_full_replay(full)

Total generated tokens checked: 255
Mismatches: 4

First mismatches:
{'turn': 1, 'offset': 42, 'replay_token': 2400, 'cached_token': 1538, 'logit_margin': 0.125}
{'turn': 1, 'offset': 83, 'replay_token': 3900, 'cached_token': 525, 'logit_margin': 0.125}
{'turn': 3, 'offset': 1, 'replay_token': 1521, 'cached_token': 614, 'logit_margin': 0.125}
{'turn': 3, 'offset': 14, 'replay_token': 4734, 'cached_token': 3728, 'logit_margin': 0.125}

DIAGNOSTIC: Cached and full-sequence greedy decoding disagree. Inspect logit margins before deciding whether this is numerical sensitivity or a correctness issue.


## [VALIDATION E.1] Investigate Cached vs. Full-Sequence Numerical Differences

Validation E found four greedy-token disagreements between BF16 cached decoding
and full-sequence replay (4 out of 255 generated tokens).

This diagnostic repeats both execution paths using a fresh FP32 model and the
exact recorded full-baseline token history.

For each original mismatch, compare:
- FP32 cached next-token prediction
- FP32 full-sequence next-token prediction
- Original BF16 cached prediction
- Logit differences and top-token margins

A mismatch disappearing in FP32 supports a numerical-precision explanation.
A mismatch persisting in FP32 requires further investigation.

In [18]:
import gc
from transformers import AutoModelForCausalLM

def investigate_validation_e(full_pipeline):
    # The four mismatches reported by Validation E.
    mismatch_locations = [
        (1, 42),
        (1, 83),
        (3, 1),
        (3, 14),
    ]

    # Convert (turn, assistant offset) into absolute token positions.
    targets = {}

    for turn_number, offset in mismatch_locations:
        turn = full_pipeline.turns[turn_number - 1]

        position = turn.assistant_logical_start + offset

        targets[position] = {
            "turn": turn_number,
            "offset": offset,
            "original_bf16_token": turn.assistant_ids[offset],
        }

    token_ids = full_pipeline.logical_token_ids
    total_length = len(token_ids)

    print("Recorded context length:", total_length)
    print("Positions to investigate:", sorted(targets))
    print("Loading temporary FP32 model...")

    fp32_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float32,
        attn_implementation="sdpa",
    ).to(DEVICE)

    fp32_model.eval()

    # Avoid TF32 approximations during the FP32 comparison.
    old_tf32 = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = False

    results = []

    try:
        with torch.inference_mode():

            # ----------------------------------------------------
            # A. Full-sequence FP32 replay
            # ----------------------------------------------------
            inputs = torch.tensor(
                [token_ids],
                dtype=torch.long,
                device=DEVICE,
            )

            full_output = fp32_model(
                input_ids=inputs,
                attention_mask=torch.ones_like(inputs),
                use_cache=False,
            )

            # Token at position p is predicted by logits at p-1.
            full_logits = {
                p: full_output.logits[0, p - 1].float().cpu()
                for p in targets
            }

            del full_output, inputs

            # ----------------------------------------------------
            # B. Cached FP32 replay, one token at a time
            # ----------------------------------------------------
            cache = None
            cached_logits = {}

            last_needed_position = max(targets)

            for position in range(last_needed_position):
                token = torch.tensor(
                    [[token_ids[position]]],
                    dtype=torch.long,
                    device=DEVICE,
                )

                output = fp32_model(
                    input_ids=token,
                    attention_mask=torch.ones(
                        (1, position + 1),
                        dtype=torch.long,
                        device=DEVICE,
                    ),
                    position_ids=torch.tensor(
                        [[position]],
                        dtype=torch.long,
                        device=DEVICE,
                    ),
                    past_key_values=cache,
                    use_cache=True,
                )

                cache = output.past_key_values

                predicted_position = position + 1

                if predicted_position in targets:
                    cached_logits[predicted_position] = (
                        output.logits[0, -1].float().cpu()
                    )

                del output

            del cache

            # ----------------------------------------------------
            # C. Compare the two FP32 execution paths
            # ----------------------------------------------------
            for position in sorted(targets):
                info = targets[position]

                a = cached_logits[position]
                b = full_logits[position]

                cached_winner = int(a.argmax())
                full_winner = int(b.argmax())

                cached_top2 = torch.topk(a, 2).values
                full_top2 = torch.topk(b, 2).values

                cached_margin = (
                    cached_top2[0] - cached_top2[1]
                ).item()

                full_margin = (
                    full_top2[0] - full_top2[1]
                ).item()

                max_logit_difference = (
                    a - b
                ).abs().max().item()

                result = {
                    "turn": info["turn"],
                    "offset": info["offset"],
                    "bf16_cached_token": info["original_bf16_token"],
                    "fp32_cached_token": cached_winner,
                    "fp32_full_token": full_winner,
                    "fp32_predictions_match": (
                        cached_winner == full_winner
                    ),
                    "cached_top2_margin": round(cached_margin, 6),
                    "full_top2_margin": round(full_margin, 6),
                    "max_logit_difference": round(
                        max_logit_difference, 6
                    ),
                }

                results.append(result)

    finally:
        torch.backends.cuda.matmul.allow_tf32 = old_tf32

        del fp32_model
        gc.collect()
        torch.cuda.empty_cache()

    # ------------------------------------------------------------
    # D. Report
    # ------------------------------------------------------------
    print("\n=== VALIDATION E.1 RESULTS ===\n")

    for result in results:
        print(
            f"Turn {result['turn']}, "
            f"token offset {result['offset']}"
        )

        print(
            "  Original BF16 cached token:",
            result["bf16_cached_token"],
        )

        print(
            "  FP32 cached token:",
            result["fp32_cached_token"],
        )

        print(
            "  FP32 full-sequence token:",
            result["fp32_full_token"],
        )

        print(
            "  FP32 predictions agree:",
            result["fp32_predictions_match"],
        )

        print(
            "  FP32 cached top-2 margin:",
            result["cached_top2_margin"],
        )

        print(
            "  FP32 full top-2 margin:",
            result["full_top2_margin"],
        )

        print(
            "  Maximum FP32 logit difference:",
            result["max_logit_difference"],
        )

        print()

    all_match = all(
        result["fp32_predictions_match"]
        for result in results
    )

    print("================================")

    if all_match:
        print(
            "RESULT: All four original mismatch positions "
            "have matching FP32 greedy predictions."
        )
        print(
            "This supports a numerical-precision explanation "
            "for the original BF16 disagreements."
        )
        print(
            "It does not establish full-sequence equivalence "
            "at every token position."
        )
    else:
        print(
            "RESULT: Some mismatches persist in FP32."
        )
        print(
            "Further investigation of cache execution "
            "and positional handling is required."
        )

    return results


if RUN_VALIDATIONS:
    validation_e1_results = investigate_validation_e(full)

Recorded context length: 406
Positions to investigate: [106, 147, 290, 303]
Loading temporary FP32 model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


=== VALIDATION E.1 RESULTS ===

Turn 1, token offset 42
  Original BF16 cached token: 1538
  FP32 cached token: 2400
  FP32 full-sequence token: 2400
  FP32 predictions agree: True
  FP32 cached top-2 margin: 0.070747
  FP32 full top-2 margin: 0.07074
  Maximum FP32 logit difference: 2.9e-05

Turn 1, token offset 83
  Original BF16 cached token: 525
  FP32 cached token: 3900
  FP32 full-sequence token: 3900
  FP32 predictions agree: True
  FP32 cached top-2 margin: 0.037785
  FP32 full top-2 margin: 0.037767
  Maximum FP32 logit difference: 2.6e-05

Turn 3, token offset 1
  Original BF16 cached token: 614
  FP32 cached token: 1521
  FP32 full-sequence token: 1521
  FP32 predictions agree: True
  FP32 cached top-2 margin: 0.154264
  FP32 full top-2 margin: 0.154245
  Maximum FP32 logit difference: 2.8e-05

Turn 3, token offset 14
  Original BF16 cached token: 3728
  FP32 cached token: 3728
  FP32 full-sequence token: 3728
  FP32 predictions agree: True
  FP32 cached top-2 margin: 0.039

## [VALIDATION E.2] All-Token FP32 Cached vs. Full-Sequence Equivalence

Compare a temporary FP32 Qwen model using two inference paths:

1. Full-sequence, non-cached replay.
2. Cached, one-token-at-a-time replay.

Both paths use the exact token history recorded by the full-cache baseline.
At every generated assistant-token position, compare the greedy prediction
and measure numerical differences in the logits.

In [19]:
# ============================================================
# VALIDATION E.2 — All-token FP32 equivalence
# ============================================================

import gc
from transformers import AutoModelForCausalLM


def validate_fp32_all_tokens(full_pipeline):
    token_ids = full_pipeline.logical_token_ids
    positions = []
    metadata = {}

    for turn in full_pipeline.turns:
        start = turn.assistant_logical_start

        for offset, expected_token in enumerate(turn.assistant_ids):
            position = start + offset

            positions.append(position)
            metadata[position] = {
                "turn": turn.number,
                "offset": offset,
                "recorded_token": int(expected_token),
            }

    assert positions, "No assistant tokens found"
    assert len(positions) == len(set(positions))
    assert min(positions) > 0
    assert max(positions) < len(token_ids)

    print("Recorded context tokens:", len(token_ids))
    print("Assistant tokens to compare:", len(positions))
    print("Loading temporary FP32 model...")

    fp32_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float32,
        attn_implementation="sdpa",
    ).to(DEVICE)

    fp32_model.eval()

    old_tf32_matmul = torch.backends.cuda.matmul.allow_tf32
    old_tf32_cudnn = torch.backends.cudnn.allow_tf32

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

    mismatches = []
    max_logit_difference = 0.0
    max_difference_position = None
    minimum_top2_margin = float("inf")

    try:
        with torch.inference_mode():

            # ----------------------------------------------------
            # A. Full-sequence FP32 forward pass
            # ----------------------------------------------------
            inputs = torch.tensor(
                [token_ids],
                dtype=torch.long,
                device=DEVICE,
            )

            full_output = fp32_model(
                input_ids=inputs,
                attention_mask=torch.ones_like(inputs),
                use_cache=False,
            )

            # Keep only logits predicting assistant tokens.
            full_logits = {
                position: full_output.logits[
                    0, position - 1
                ].float().cpu()
                for position in positions
            }

            del full_output, inputs

            # ----------------------------------------------------
            # B. FP32 cached replay, one token at a time
            # ----------------------------------------------------
            cache = None
            target_positions = set(positions)

            # To predict token p, process tokens through p-1.
            for index in range(max(positions)):
                input_id = torch.tensor(
                    [[token_ids[index]]],
                    dtype=torch.long,
                    device=DEVICE,
                )

                output = fp32_model(
                    input_ids=input_id,
                    attention_mask=torch.ones(
                        (1, index + 1),
                        dtype=torch.long,
                        device=DEVICE,
                    ),
                    position_ids=torch.tensor(
                        [[index]],
                        dtype=torch.long,
                        device=DEVICE,
                    ),
                    past_key_values=cache,
                    use_cache=True,
                )

                cache = output.past_key_values
                predicted_position = index + 1

                if predicted_position in target_positions:
                    cached_logits = (
                        output.logits[0, -1].float().cpu()
                    )
                    reference_logits = full_logits[
                        predicted_position
                    ]

                    assert torch.isfinite(cached_logits).all()
                    assert torch.isfinite(reference_logits).all()

                    difference = (
                        cached_logits - reference_logits
                    ).abs().max().item()

                    if difference > max_logit_difference:
                        max_logit_difference = difference
                        max_difference_position = predicted_position

                    cached_prediction = int(
                        cached_logits.argmax()
                    )
                    full_prediction = int(
                        reference_logits.argmax()
                    )

                    cached_top2 = torch.topk(
                        cached_logits, 2
                    ).values
                    full_top2 = torch.topk(
                        reference_logits, 2
                    ).values

                    cached_margin = (
                        cached_top2[0] - cached_top2[1]
                    ).item()

                    full_margin = (
                        full_top2[0] - full_top2[1]
                    ).item()

                    minimum_top2_margin = min(
                        minimum_top2_margin,
                        cached_margin,
                        full_margin,
                    )

                    if cached_prediction != full_prediction:
                        info = metadata[predicted_position]

                        mismatches.append({
                            "turn": info["turn"],
                            "offset": info["offset"],
                            "position": predicted_position,
                            "cached_prediction": cached_prediction,
                            "full_prediction": full_prediction,
                            "max_logit_difference": difference,
                            "cached_top2_margin": cached_margin,
                            "full_top2_margin": full_margin,
                        })

                del output

            del cache

    finally:
        torch.backends.cuda.matmul.allow_tf32 = old_tf32_matmul
        torch.backends.cudnn.allow_tf32 = old_tf32_cudnn

        del fp32_model
        gc.collect()
        torch.cuda.empty_cache()

    # ------------------------------------------------------------
    # C. Results
    # ------------------------------------------------------------
    result = {
        "checked_tokens": len(positions),
        "mismatch_count": len(mismatches),
        "mismatches": mismatches,
        "max_logit_difference": max_logit_difference,
        "max_difference_position": max_difference_position,
        "minimum_top2_margin": minimum_top2_margin,
    }

    print("\n=== VALIDATION E.2 RESULTS ===")
    print("Generated-token positions checked:", len(positions))
    print("FP32 prediction mismatches:", len(mismatches))
    print(
        "Maximum absolute FP32 logit difference:",
        f"{max_logit_difference:.8f}",
    )
    print(
        "Position of maximum difference:",
        max_difference_position,
    )
    print(
        "Minimum FP32 top-2 margin:",
        f"{minimum_top2_margin:.8f}",
    )

    if mismatches:
        print("\nFirst mismatches:")
        for mismatch in mismatches[:10]:
            print(mismatch)

    assert not mismatches, (
        f"Validation E.2 failed: "
        f"{len(mismatches)}/{len(positions)} FP32 "
        "cached/full greedy predictions disagree"
    )

    print(
        "\nPASS: FP32 cached and full-sequence greedy "
        "predictions agree at every recorded assistant-token position"
    )

    return result


if RUN_VALIDATIONS:
    validation_e2_result = validate_fp32_all_tokens(full)

Recorded context tokens: 406
Assistant tokens to compare: 255
Loading temporary FP32 model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


=== VALIDATION E.2 RESULTS ===
Generated-token positions checked: 255
FP32 prediction mismatches: 0
Maximum absolute FP32 logit difference: 0.00009346
Position of maximum difference: 226
Minimum FP32 top-2 margin: 0.00776291

PASS: FP32 cached and full-sequence greedy predictions agree at every recorded assistant-token position


## [VALIDATION F] 100% Retention Must Equal the Full-Cache Pipeline
Run a **third pipeline** over the same five user turns, with `retention_ratio=1.0`. It still captures queries, calculates scores, and enters the compaction code at every eligible turn, but removes **zero** entries.

In [20]:
def validate_100_percent_retention(full_pipeline, full_outputs, user_turns):
    untouched = DelayedTEPipeline(
        model, tokenizer, mode="fixed", retention_ratio=1.0,
        include_eos_proxy=INCLUDE_EOS_PROXY,
        compaction_validator=validate_compaction_gather,
    )
    for index, user_text in enumerate(user_turns):
        result = untouched.run_turn(user_text)
        expected = full_outputs[index]
        assert result["assistant_ids"] == expected["assistant_ids"], (
            f"100% retention changed generation at T{index + 1}"
        )
        assert result["physical_tokens"] == expected["physical_tokens"]
        assert result["logical_tokens"] == expected["logical_tokens"]
        if index >= 1:
            assert untouched.events[-1]["removed_tokens"] == 0
            assert untouched.events[-1]["retained_tokens"] == (
                untouched.events[-1]["original_tokens"]
            )

    assert untouched.logical_token_ids == full_pipeline.logical_token_ids
    assert untouched.logical_position == full_pipeline.logical_position
    assert len(untouched.events) == len(user_turns) - 1
    assert untouched.validated_compactions == len(untouched.events)
    assert cache_length(untouched.cache) == cache_length(full_pipeline.cache)
    for layer_idx in range(NUM_LAYERS):
        for name in ("keys", "values"):
            actual = getattr(untouched.cache.layers[layer_idx], name)
            expected = getattr(full_pipeline.cache.layers[layer_idx], name)
            assert torch.equal(actual, expected), (
                f"100% cache differs at layer {layer_idx}, tensor {name}"
            )
    validate_recorded_chat_boundaries(untouched)
    print("PASS: 100% retention exactly matches full-cache generation "
          "and every final K/V entry across all five turns")
    return untouched


if RUN_VALIDATIONS:
    fixed_100 = validate_100_percent_retention(full, full_results, USER_TURNS)


PASS: exact K/V gather, T2 -> T1, 0 entries removed per KV head/layer
PASS: exact K/V gather, T3 -> T2, 0 entries removed per KV head/layer
PASS: exact K/V gather, T4 -> T3, 0 entries removed per KV head/layer
PASS: exact K/V gather, T5 -> T4, 0 entries removed per KV head/layer
PASS (fixed): every user/assistant logical boundary and Qwen chat-template structure
PASS: 100% retention exactly matches full-cache generation and every final K/V entry across all five turns


## [VALIDATION GATE] Fixed-Ratio TE Correctness Checklist
All previous assertions must execute without error; this cell summarizes the three persistent bookkeeping requirements, rather than repeating the numeric scoring and SDPA tests. The five gates are: **(1)** exact K/V selection and unchanged historical K/V after a probe, **(2)** full-context causal normalization, **(3)** captured post-RoPE Q/GQA agreement with SDPA, **(4)** canonical chat tokens plus recorded assistant boundaries, and **(5)** no-compaction equivalence including 100% retention and ordinary full-context replay.

In [21]:
# ============================================================
# VALIDATION G — Final Correctness Summary
# ============================================================

if RUN_VALIDATIONS:

    print("=" * 65)
    print("VALIDATION G — FINAL CORRECTNESS SUMMARY")
    print("=" * 65)

    expected_events = len(USER_TURNS) - 1

    # --------------------------------------------------------
    # G.1 — Compaction validation coverage
    # --------------------------------------------------------
    assert len(fixed.events) == expected_events
    assert fixed.validated_compactions == expected_events

    assert len(fixed_100.events) == expected_events
    assert fixed_100.validated_compactions == expected_events

    print(
        f"PASS — G.1: All {expected_events} compaction events "
        "validated in both fixed-ratio and 100% retention runs"
    )

    # --------------------------------------------------------
    # G.2 — Final cache and position bookkeeping
    # --------------------------------------------------------
    for pipeline in (full, fixed, fixed_100):
        assert_cache_shapes(pipeline.cache)

        assert pipeline.logical_position == len(
            pipeline.logical_token_ids
        )

    assert full.logical_position == cache_length(full.cache)

    assert (
        fixed.logical_position - cache_length(fixed.cache)
        == sum(
            event["removed_tokens"]
            for event in fixed.events
        )
    )

    assert not fixed.turns[-1].compacted
    assert all(
        turn.compacted
        for turn in fixed.turns[:-1]
    )

    for turn in fixed.turns:
        assert (
            turn.assistant_end - turn.assistant_start
            == turn.retained_length
        )

    print(
        "PASS — G.2: Final cache shapes, logical positions, "
        "and compaction bookkeeping"
    )

    # --------------------------------------------------------
    # G.3 — 100% retention equivalence
    # --------------------------------------------------------
    assert all(
        event["removed_tokens"] == 0
        for event in fixed_100.events
    )

    assert (
        full.logical_token_ids
        == fixed_100.logical_token_ids
    )

    assert (
        cache_length(full.cache)
        == cache_length(fixed_100.cache)
    )

    for layer_idx in range(NUM_LAYERS):
        for tensor_name in ("keys", "values"):
            actual = getattr(
                fixed_100.cache.layers[layer_idx],
                tensor_name,
            )

            expected = getattr(
                full.cache.layers[layer_idx],
                tensor_name,
            )

            assert torch.equal(actual, expected), (
                f"100% retention differs from full cache: "
                f"layer {layer_idx}, {tensor_name}"
            )

    print(
        "PASS — G.3: 100% retention matches full-cache "
        "token history and final K/V tensors"
    )

    # --------------------------------------------------------
    # E — BF16 replay result
    # --------------------------------------------------------
    print("-" * 65)
    print("REPLAY VALIDATION RESULTS")
    print("-" * 65)

    if "replay_mismatches" in globals():
        bf16_count = len(replay_mismatches)

        if bf16_count == 0:
            print(
                "PASS — E: BF16 cached/full greedy replay"
            )
        else:
            print(
                f"NOT PASSED — E: {bf16_count} BF16 "
                "cached/full greedy-token mismatches"
            )
    else:
        print(
            "NOT VERIFIED — E: BF16 replay result unavailable"
        )

    # --------------------------------------------------------
    # E.1 — Targeted FP32 comparison
    # --------------------------------------------------------
    if "validation_e1_results" in globals():
        e1_results = validation_e1_results

        e1_checked = len(e1_results)
        e1_matched = sum(
            item["fp32_predictions_match"]
            for item in e1_results
        )

        if e1_checked > 0 and e1_matched == e1_checked:
            print(
                f"PASS — E.1: FP32 agreement at all "
                f"{e1_checked} investigated BF16 mismatch positions"
            )
        else:
            print(
                f"NOT PASSED — E.1: FP32 agreement at "
                f"{e1_matched}/{e1_checked} investigated positions"
            )
    else:
        print(
            "NOT VERIFIED — E.1: Targeted FP32 results unavailable"
        )

    # --------------------------------------------------------
    # E.2 — All-token FP32 equivalence
    # --------------------------------------------------------
    assert "validation_e2_result" in globals(), (
        "Validation E.2 has not completed successfully"
    )

    e2 = validation_e2_result

    expected_assistant_tokens = sum(
        len(turn.assistant_ids)
        for turn in full.turns
    )

    assert (
        e2["checked_tokens"]
        == expected_assistant_tokens
    ), "E.2 did not cover every generated assistant token"

    assert e2["mismatch_count"] == 0, (
        "E.2 detected FP32 cached/full prediction mismatches"
    )

    print(
        f"PASS — E.2: FP32 cached/full greedy predictions "
        f"match at all {e2['checked_tokens']} generated-token positions"
    )

    print(
        "          Maximum FP32 logit difference:",
        f"{e2['max_logit_difference']:.8f}",
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------
    print("-" * 65)

    print(
        "Earlier validations A–D and F are not independently "
        "rerun by this summary cell."
    )

    print(
        "Their original validation cells must have "
        "completed successfully."
    )

    print("-" * 65)
    print(
        "RESULT: Final-state compaction checks and "
        "all-token FP32 replay equivalence passed."
    )

    if (
        "replay_mismatches" in globals()
        and len(replay_mismatches) > 0
    ):
        print(
            "NOTE: Strict BF16 replay remains non-identical. "
            "The FP32 results support numerical sensitivity "
            "as the explanation."
        )

else:
    print(
        "VALIDATION G NOT RUN: Set RUN_VALIDATIONS=True."
    )

VALIDATION G — FINAL CORRECTNESS SUMMARY
PASS — G.1: All 4 compaction events validated in both fixed-ratio and 100% retention runs
PASS — G.2: Final cache shapes, logical positions, and compaction bookkeeping
PASS — G.3: 100% retention matches full-cache token history and final K/V tensors
-----------------------------------------------------------------
REPLAY VALIDATION RESULTS
-----------------------------------------------------------------
NOT PASSED — E: 4 BF16 cached/full greedy-token mismatches
PASS — E.1: FP32 agreement at all 4 investigated BF16 mismatch positions
PASS — E.2: FP32 cached/full greedy predictions match at all 255 generated-token positions
          Maximum FP32 logit difference: 0.00009346
-----------------------------------------------------------------
Earlier validations A–D and F are not independently rerun by this summary cell.
Their original validation cells must have completed successfully.
----------------------------------------------------------------

Implemented a generalized one-turn-delayed, fixed-ratio token eviction pipeline for Qwen2.5-0.5B-Instruct. Across a five-turn conversation, the system successfully performed four validated compactions and reduced final K/V-cache storage by 28.3%. The resulting responses differed from full-cache inference on two turns, while both configurations also exhibited a shared factual error. These results establish implementation correctness and demonstrate memory savings, but do not yet establish reliable preservation of factual information.